# 03 — Machine Learning Models

Train and evaluate the irrigation classification and irrigation-quantity regression models.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from pathlib import Path

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)


## 2. Load processed irrigation dataset

In [ ]:
DATA_PATH = Path("../data/processed/irrigation_dataset.csv")
model_df = pd.read_csv(DATA_PATH)
model_df["Date"] = pd.to_datetime(model_df["Date"])


## 3. Historical features

In [ ]:
model_df = model_df.sort_values("Date").reset_index(drop=True)

model_df["ET0_lag1"] = model_df["ET0"].shift(1)
model_df["ET0_3day"] = model_df["ET0"].rolling(3).sum().shift(1)
model_df["ET0_7day"] = model_df["ET0"].rolling(7).sum().shift(1)

model_df["Rainfall_3day"] = model_df["Rainfall"].rolling(3).sum().shift(1)
model_df["Rainfall_7day"] = model_df["Rainfall"].rolling(7).sum().shift(1)

model_df["ETc_3day"] = model_df["ETc"].rolling(3).sum().shift(1)
model_df["ETc_7day"] = model_df["ETc"].rolling(7).sum().shift(1)


## 4. Classification

In [ ]:
classification_features = [
    "Tmean","Max_Temperature","Min_Temperature",
    "Relative_Humidity","Rainfall","Wind_Speed",
    "Solar_Radiation","ET0","ET0_lag1","ET0_3day","ET0_7day",
    "Rainfall_3day","Rainfall_7day","ETc_3day","ETc_7day",
    "crop_age","Kc","field_capacity","wilting_point","root_depth"
]

class_df = model_df.dropna(
    subset=classification_features + ["irrigate"]
).copy()

X = class_df[classification_features]
y = class_df["irrigate"].astype(int)

split = int(len(class_df) * 0.80)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print("Train:", X_train.shape, "Test:", X_test.shape)
print(y.value_counts())


## 5. Balance training data with SMOTE

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(y_train_smote.value_counts())


## 6. Train classification model

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
classifier.fit(X_train_smote, y_train_smote)

y_pred = classifier.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


## 7. Feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "feature": classification_features,
    "importance": classifier.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance)

top = feature_importance.head(15).iloc[::-1]
plt.figure(figsize=(9,6))
plt.barh(top["feature"], top["importance"])
plt.xlabel("Importance")
plt.title("Top Features for Irrigation Prediction")
plt.tight_layout()
plt.show()


## 8. Regression — irrigation days only

In [ ]:
reg_features = [
    "Tmean","Max_Temperature","Min_Temperature",
    "Relative_Humidity","Rainfall","Wind_Speed","Solar_Radiation",
    "ET0","ET0_lag1","ET0_3day","ET0_7day",
    "Rainfall_3day","Rainfall_7day","ETc_3day","ETc_7day",
    "crop_age","Kc","field_capacity","wilting_point","root_depth"
]

reg_df = model_df[model_df["irrigate"] == True].dropna(
    subset=reg_features + ["net_irrigation"]
).copy()

X_reg = reg_df[reg_features]
y_reg = reg_df["net_irrigation"]

split_reg = int(len(reg_df) * 0.80)

X_train_reg, X_test_reg = X_reg.iloc[:split_reg], X_reg.iloc[split_reg:]
y_train_reg, y_test_reg = y_reg.iloc[:split_reg], y_reg.iloc[split_reg:]

print("Regression train:", len(X_train_reg))
print("Regression test:", len(X_test_reg))


## 9. Compare regression models

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300, max_depth=10, min_samples_leaf=2, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300, max_depth=10, min_samples_leaf=2, random_state=42
    )
}

rows = []

for name, model in models.items():
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)

    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test_reg, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_reg, pred)),
        "R2": r2_score(y_test_reg, pred)
    })

results_df = pd.DataFrame(rows).sort_values("R2", ascending=False)
display(results_df)


## 10. Train final regression model

In [ ]:
final_regressor = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42
)

final_regressor.fit(X_train_reg, y_train_reg)
y_pred_reg = final_regressor.predict(X_test_reg)

print(f"MAE : {mean_absolute_error(y_test_reg, y_pred_reg):.3f} mm")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred_reg)):.3f} mm")
print(f"R2  : {r2_score(y_test_reg, y_pred_reg):.3f}")


## 11. Actual vs predicted

In [ ]:
comparison = pd.DataFrame({
    "Date": reg_df.iloc[split_reg:]["Date"].values,
    "Actual": y_test_reg.values,
    "Predicted": y_pred_reg
})
display(comparison.round(2))

plt.figure(figsize=(7,6))
plt.scatter(comparison["Actual"], comparison["Predicted"])
m1 = min(comparison["Actual"].min(), comparison["Predicted"].min())
m2 = max(comparison["Actual"].max(), comparison["Predicted"].max())
plt.plot([m1,m2],[m1,m2],"--")
plt.xlabel("Actual irrigation (mm)")
plt.ylabel("Predicted irrigation (mm)")
plt.title("Actual vs Predicted Irrigation")
plt.tight_layout()
plt.show()


## 12. Save models

In [ ]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(classifier, MODEL_DIR / "irrigation_classifier.pkl")
joblib.dump(classification_features, MODEL_DIR / "classification_features.pkl")
joblib.dump(final_regressor, MODEL_DIR / "irrigation_regressor.pkl")
joblib.dump(reg_features, MODEL_DIR / "regression_features.pkl")

print("Saved trained models and feature lists to", MODEL_DIR)


## 13. Final interpretation

The classification model predicts whether irrigation is required. The regression model predicts net irrigation quantity when irrigation is required. The current system is a proof of concept because the irrigation targets are generated from the agronomic water-balance framework; field validation with observed irrigation and soil-moisture data is the next major step.